# Silver project

## Purpose

Create the canonical project-level Silver table used by future project-region matching, flood-control reconciliation, portfolio analytics, delivery analytics, and Gold project dimensions and facts.

This table consolidates component-level evidence while preserving conflicts and data-quality findings.

One output row represents one canonical project for the selected source snapshot and Silver processing rule set.

# 1. Read component-level project evidence

The project table is built entirely from `02-silver`.silver_dpwh_project_component.

Bronze data is not re-read and re-cleaned.

This ensures a single source of truth for parsing, standardization, quality controls, lineage, and mapping results.

Expected validation:

- All project inputs originate from the component table
- Source lineage remains intact
- Project accounting reconciles to component accounting
``

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW project_component_input AS
SELECT
    contract_id,
    source_system,

    project_description_raw,

    category_raw,
    standardized_sector,
    standardized_category,
    taxonomy_version,

    status_raw,
    standardized_status,
    status_group,
    status_mapping_version,
    status_mapping_state,

    reported_budget_raw,
    reported_budget_pesos,

    progress_raw,
    physical_progress_pct,

    start_date_raw,
    start_date_parsed,

    completion_date_raw,
    completion_date_parsed,

    latitude_raw,
    latitude_parsed,

    longitude_raw,
    longitude_parsed,

    coordinate_status,
    infra_year,
    contractor,
    source_of_funds,
    reported_region,

    _source_snapshot_id,
    _ingest_run_id,
    _source_modified_at

FROM `02-silver`.silver_dpwh_project_component;

# 2. Resolve project-level attributes conservatively

Project-level values are only published when component-level evidence resolves consistently.

When exactly one meaningful value exists, the value becomes the canonical project value.

When no meaningful value exists, the value remains unresolved.

When multiple conflicting values exist, the conflict remains visible rather than being resolved automatically.

Expected validation:

- Conflicts are reported explicitly
- No arbitrary source row is selected
- Attribute resolution remains deterministic


In [ ]:
%sql
SELECT ASSERT_TRUE(
    COUNT_IF(
        contract_id IS NULL
        OR TRIM(contract_id) = ''
    ) = 0,
    'STOP: project consolidation cannot safely group missing Contract IDs'
)
FROM `02-silver`.silver_dpwh_project_component;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_attribute_resolution AS
SELECT
    contract_id,

    COUNT(*) AS component_count,

    COUNT(DISTINCT NULLIF(TRIM(project_description_raw), ''))
        AS distinct_description_count,

    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(project_description_raw), '')) = 1
            THEN MAX(NULLIF(TRIM(project_description_raw), ''))
        ELSE NULL
    END AS project_description_raw,

    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(project_description_raw), '')) = 0
            THEN 'MISSING'
        WHEN COUNT(DISTINCT NULLIF(TRIM(project_description_raw), '')) = 1
            THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS description_resolution_status,

    COUNT(DISTINCT NULLIF(TRIM(category_raw), ''))
        AS distinct_category_count,

    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(category_raw), '')) = 1
            THEN MAX(NULLIF(TRIM(category_raw), ''))
        ELSE NULL
    END AS category_raw,

    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(category_raw), '')) = 0
            THEN 'MISSING'
        WHEN COUNT(DISTINCT NULLIF(TRIM(category_raw), '')) = 1
            THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS category_resolution_status,

    COUNT(DISTINCT NULLIF(TRIM(status_raw), ''))
        AS distinct_status_count,

    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(status_raw), '')) = 1
            THEN MAX(NULLIF(TRIM(status_raw), ''))
        ELSE NULL
    END AS status_raw,

    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(status_raw), '')) = 0
            THEN 'MISSING'
        WHEN COUNT(DISTINCT NULLIF(TRIM(status_raw), '')) = 1
            THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS status_resolution_status,

    COUNT(DISTINCT physical_progress_pct)
        AS distinct_progress_count,

    CASE
        WHEN COUNT(DISTINCT physical_progress_pct) = 1
            THEN MAX(physical_progress_pct)
        ELSE NULL
    END AS physical_progress_pct,

    CASE
        WHEN COUNT(DISTINCT physical_progress_pct) = 0
            THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT physical_progress_pct) = 1
            THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS progress_resolution_status,

    COUNT(DISTINCT start_date_parsed)
        AS distinct_start_date_count,

    CASE
        WHEN COUNT(DISTINCT start_date_parsed) = 1
            THEN MAX(start_date_parsed)
        ELSE NULL
    END AS start_date,

    CASE
        WHEN COUNT(DISTINCT start_date_parsed) = 0
            THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT start_date_parsed) = 1
            THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS start_date_resolution_status,

    COUNT(DISTINCT completion_date_parsed)
        AS distinct_completion_date_count,

    CASE
        WHEN COUNT(DISTINCT completion_date_parsed) = 1
            THEN MAX(completion_date_parsed)
        ELSE NULL
    END AS completion_date,

    CASE
        WHEN COUNT(DISTINCT completion_date_parsed) = 0
            THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT completion_date_parsed) = 1
            THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS completion_date_resolution_status,

    COUNT(DISTINCT reported_region)
        AS distinct_reported_region_count,

    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(reported_region), '')) = 1
            THEN MAX(NULLIF(TRIM(reported_region), ''))
        ELSE NULL
    END AS reported_region,

    COUNT(DISTINCT contractor)
        AS distinct_contractor_count,

    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(contractor), '')) = 1
            THEN MAX(NULLIF(TRIM(contractor), ''))
        ELSE NULL
    END AS contractor,

    COUNT(DISTINCT source_of_funds)
        AS distinct_funding_source_count,

    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(source_of_funds), '')) = 1
            THEN MAX(NULLIF(TRIM(source_of_funds), ''))
        ELSE NULL
    END AS source_of_funds,

    COUNT(DISTINCT _source_snapshot_id)
        AS distinct_snapshot_count,

    CASE
        WHEN COUNT(DISTINCT _source_snapshot_id) = 1
            THEN MAX(_source_snapshot_id)
        ELSE NULL
    END AS source_snapshot_id,

    COUNT(DISTINCT _ingest_run_id)
        AS distinct_ingest_run_count,

    CASE
    WHEN COUNT(DISTINCT _ingest_run_id) = 1
        THEN MAX(_ingest_run_id)
    ELSE NULL
END AS source_ingest_run_id,

COUNT(DISTINCT infra_year)
    AS distinct_infra_year_count,

CASE
    WHEN COUNT(DISTINCT infra_year) = 1
        THEN MAX(infra_year)
    ELSE NULL
END AS infra_year,

COUNT(DISTINCT latitude_parsed)
    AS distinct_latitude_count,

CASE
    WHEN COUNT(DISTINCT latitude_parsed) = 1
        THEN MAX(latitude_parsed)
    ELSE NULL
END AS latitude_parsed,

COUNT(DISTINCT longitude_parsed)
    AS distinct_longitude_count,

CASE
    WHEN COUNT(DISTINCT longitude_parsed) = 1
        THEN MAX(longitude_parsed)
    ELSE NULL
END AS longitude_parsed,

COUNT(DISTINCT standardized_category)
    AS distinct_standardized_category_count,

CASE
    WHEN COUNT(DISTINCT standardized_category) = 1
        THEN MAX(standardized_category)
    ELSE NULL
END AS standardized_category,

COUNT(DISTINCT standardized_status)
    AS distinct_standardized_status_count,

CASE
    WHEN COUNT(DISTINCT standardized_status) = 1
        THEN MAX(standardized_status)
    ELSE NULL
END AS standardized_status,

COUNT(DISTINCT taxonomy_version)
    AS distinct_taxonomy_version_count,

CASE
    WHEN COUNT(DISTINCT taxonomy_version) = 1
        THEN MAX(taxonomy_version)
    ELSE NULL
END AS taxonomy_version,

COUNT(DISTINCT status_mapping_version)
    AS distinct_status_mapping_version_count,

CASE
    WHEN COUNT(DISTINCT status_mapping_version) = 1
        THEN MAX(status_mapping_version)
    ELSE NULL
END AS status_mapping_version,
COUNT(DISTINCT NULLIF(TRIM(source_system), ''))
    AS distinct_source_system_count,

CASE
    WHEN COUNT(DISTINCT NULLIF(TRIM(source_system), '')) = 1
        THEN MAX(NULLIF(TRIM(source_system), ''))
    ELSE NULL
END AS source_system,

COUNT(DISTINCT NULLIF(TRIM(standardized_sector), ''))
    AS distinct_standardized_sector_count,

CASE
    WHEN COUNT(DISTINCT NULLIF(TRIM(standardized_sector), '')) = 1
        THEN MAX(NULLIF(TRIM(standardized_sector), ''))
    ELSE NULL
END AS standardized_sector,

COUNT(DISTINCT NULLIF(TRIM(status_group), ''))
    AS distinct_status_group_count,

CASE
    WHEN COUNT(DISTINCT NULLIF(TRIM(status_group), '')) = 1
        THEN MAX(NULLIF(TRIM(status_group), ''))
    ELSE NULL
END AS status_group,

COUNT(DISTINCT NULLIF(TRIM(status_mapping_state), ''))
    AS distinct_status_mapping_state_count,

CASE
    WHEN COUNT(DISTINCT NULLIF(TRIM(status_mapping_state), '')) = 1
        THEN MAX(NULLIF(TRIM(status_mapping_state), ''))
    ELSE NULL
END AS status_mapping_state,
MAX(_source_modified_at) AS source_modified_at

FROM project_component_input
GROUP BY contract_id;


# 3. Protect against budget double counting

Repeated component rows must not automatically increase project investment values.

The project budget represents the reported project or contract budget and does not imply actual payments or disbursements.

Repeated identical values may resolve to a single canonical budget value.

Multiple conflicting budget values remain unresolved until a governed business rule is approved.

Expected validation:

- Repeated values are not blindly summed
- Conflicting values remain visible
- Budget resolution status is reported
- Project-level investment remains explainable

In [ ]:

%sql
CREATE OR REPLACE TEMPORARY VIEW project_budget_resolution AS
SELECT
    contract_id,

    COUNT(DISTINCT reported_budget_pesos)
        AS distinct_budget_value_count,

    COUNT(reported_budget_pesos)
        AS valid_budget_component_count,

    CASE
        WHEN COUNT(DISTINCT reported_budget_pesos) = 1
            THEN MAX(reported_budget_pesos)
        ELSE NULL
    END AS reported_budget_pesos,

    CASE
        WHEN COUNT(DISTINCT reported_budget_pesos) = 0
            THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT reported_budget_pesos) = 1
             AND COUNT(reported_budget_pesos) = 1
            THEN 'RESOLVED_SINGLE'
        WHEN COUNT(DISTINCT reported_budget_pesos) = 1
             AND COUNT(reported_budget_pesos) > 1
            THEN 'RESOLVED_REPEATED_IDENTICAL'
        ELSE 'CONFLICT'
    END AS budget_resolution_status

FROM project_component_input
GROUP BY contract_id;


# 4. Publish one project-level record

The output grain is one canonical project per selected source snapshot and Silver processing rule set.

The table preserves:

- Project-level attributes
- Conflict indicators
- Delivery-rule fields
- Mapping results
- Full source lineage
- Deterministic processing identity

Expected validation:

- One row per project grain
- No duplicate project identities
- Component accounting reconciles to project accounting
- Project outputs remain traceable back to source evidence

In [ ]:
%sql
CREATE OR REPLACE TABLE `02-silver`.silver_project AS
SELECT
    a.contract_id,
    a.component_count,

    a.project_description_raw,
    a.description_resolution_status,
    a.distinct_description_count,

    a.category_raw,
    a.category_resolution_status,
    a.distinct_category_count,

    a.standardized_sector,
    a.standardized_category,
    a.taxonomy_version,

    a.status_raw,
    a.status_resolution_status,
    a.distinct_status_count,

    a.standardized_status,
    a.status_group,
    a.status_mapping_version,
    a.status_mapping_state,

    a.physical_progress_pct,
    a.progress_resolution_status,
    a.distinct_progress_count,

    a.start_date,
    a.start_date_resolution_status,
    a.distinct_start_date_count,

    a.completion_date,
    a.completion_date_resolution_status,
    a.distinct_completion_date_count,

    a.latitude_parsed,
    a.longitude_parsed,
    a.infra_year,

    a.reported_region,
    a.distinct_reported_region_count,

    a.contractor,
    a.distinct_contractor_count,

    a.source_of_funds,
    a.distinct_funding_source_count,

    b.reported_budget_pesos,
    b.budget_resolution_status,
    b.distinct_budget_value_count,
    b.valid_budget_component_count,

    CASE
        WHEN a.progress_resolution_status = 'RESOLVED'
         AND a.physical_progress_pct = 0
            THEN TRUE
        WHEN a.progress_resolution_status = 'RESOLVED'
            THEN FALSE
        ELSE NULL
    END AS zero_progress_flag,

    CAST(NULL AS BOOLEAN) AS stalled_flag,
    CAST(NULL AS BOOLEAN) AS long_running_flag,

    CASE
        WHEN a.progress_resolution_status <> 'RESOLVED'
            THEN 'UNRESOLVED_PROGRESS'
        ELSE 'ZERO_PROGRESS_RULE_EVALUATED'
    END AS zero_progress_rule_status,

    'STATUS_MAPPING_REQUIRED' AS stalled_rule_status,
    'SOURCE_SNAPSHOT_DATE_RULE_REQUIRED' AS long_running_rule_status,

    a.source_snapshot_id,
    a.source_ingest_run_id,
    a.source_modified_at,

    SHA2(
    CONCAT_WS(
        '|',
        'dpwh',
        COALESCE(a.contract_id, '')
    ),
    256
) AS project_key,

    SHA2(
        CONCAT_WS(
            '||',
            COALESCE(a.source_snapshot_id, ''),
            'silver_project_v1'
        ),
        256
    ) AS run_id,

    'silver_project_v1' AS transformation_rule_version

FROM project_attribute_resolution AS a
LEFT JOIN project_budget_resolution AS b
    ON a.contract_id = b.contract_id;

# Summary

Created `02-silver`.silver_project at one canonical project per selected source snapshot and Silver run.

Component-level evidence remains available in `02-silver`.silver_dpwh_project_component. Conflicting project attributes remain visible rather than being automatically resolved. Budget protection prevents unsupported double counting of repeated source records.


Next dependency: `02-silver`.silver_flood_control_component.